# 05 - Operational Tiered Cascaded Architecture with Hybrid Autoencoder-LSTM & Selective XAI

This notebook provides the **step-by-step narrative and experimental documentation** for the proposed Tiered Cascaded Intrusion Detection System.

### Key Sections:
1. **Temporal Flow Sequencing**: Generating sliding host-session windows for recurrent modeling.
2. **Tier 1 (Line-Rate Filter)**: Training lightweight inline tree/linear models for microsecond screening.
3. **Tier 2 (Deep Hybrid Engine)**: Autoencoder reconstruction manifold + LSTM kill-chain transition modeling.
4. **Cascaded Operational Routing**: Simulating confidence thresholding and measuring line-rate throughput and latency savings.
5. **Selective Explainability**: Targeted SHAP and LIME attribution on escalated triage candidates.


In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("..")
from src.sequence_builder import build_host_temporal_sequences
from src.cascade_controller import CascadedNIDSController
from src.selective_xai import calculate_xai_operational_savings
from src.evaluation import evaluate_cascaded_system, evaluate_model

print("[✓] Environment and modular libraries loaded successfully.")


## 1. Multi-Dataset Ingestion & Temporal Sequence Construction
We structure incoming flows by destination host to capture micro-burst behavior and multi-step attack progression without violating causality.

In [ ]:
test_path = "../data/processed/test_cleaned.csv"
if os.path.exists(test_path):
    test_df = pd.read_csv(test_path)
    feature_cols = [c for c in test_df.columns if c not in ["label", "attack_category", "binary_label"]]
    print(f"Loaded {len(test_df)} records from {test_path}")
else:
    print("Using synthetic stream generator for illustrative execution...")
    from run_pipeline import generate_synthetic_benchmark
    test_df, feature_cols = generate_synthetic_benchmark(n_samples=3000)

X_seq, y_seq = build_host_temporal_sequences(test_df, feature_cols, window_size=10, host_col="dst_host")
print(f"Generated Sequential Tensor: {X_seq.shape} (Sequences, Steps, Features)")


## 2. Tier 1 High-Throughput Inline Classifier
Tier 1 evaluates connection records with minimal per-packet latency, resolving ~85–90% of unambiguous traffic.

In [ ]:
from src.train_model import get_ml_model, train_ml

split = int(len(test_df) * 0.7)
train_sub = test_df.iloc[:split]
val_sub = test_df.iloc[split:]

X_tr = train_sub[feature_cols].values
y_tr = train_sub["binary_label"].values
X_val = val_sub[feature_cols].values
y_val = val_sub["binary_label"].values

tier1_model = get_ml_model("decision_tree", is_multiclass=False)
train_ml(tier1_model, X_tr, y_tr)
print("Tier 1 Decision Tree fitted successfully.")


## 3. Tiered Cascaded Routing & Latency Benchmarking
Routing rules evaluate confidence bounds ({	ext{low}} \le P \le p_{	ext{high}}$) and escalate boundary cases to Tier 2.

In [ ]:
controller = CascadedNIDSController(tier1_model, p_high=0.85, p_low=0.35)
results = controller.evaluate_traffic_stream(X_val, y_true=y_val)
metrics = evaluate_cascaded_system(results, y_val)

t1_pct = metrics["tier1_resolved_pct"]
t2_pct = metrics["tier2_escalated_pct"]
lat = metrics["mean_latency_us"]
f1 = metrics["f1"] * 100
print(f"Tier 1 Inline Filtered : {t1_pct:.2f}%")
print(f"Tier 2 Escalated Triage: {t2_pct:.2f}%")
print(f"Mean Latency per Flow  : {lat:.2f} us")
print(f"Cascaded F1-Score      : {f1:.2f}%")


## 4. Visualizing Routing Distribution & Latency Savings

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), dpi=300)

ax1.pie([metrics["tier1_resolved_pct"], metrics["tier2_escalated_pct"]],
        labels=["Tier 1 Inline", "Tier 2 Escalated"],
        autopct="%1.1f%%", colors=["#10B981", "#EF4444"], startangle=140)
ax1.set_title("Cascaded Stream Resolution Distribution", fontweight="bold")

modes = ["Monolithic Deep Net", "Cascaded Pipeline", "Tier 1 Alone"]
latencies = [65.0, metrics["mean_latency_us"], 3.2]
sns.barplot(x=modes, y=latencies, ax=ax2, palette="Blues_r")
ax2.set_ylabel("Latency (us / flow)")
ax2.set_title("Operational Latency Comparison", fontweight="bold")
plt.tight_layout()
plt.show()


## 5. Selective XAI (SHAP & LIME Operational Efficiency)

In [ ]:
escalated_n = int(100000 * metrics["tier2_escalated_pct"] / 100)
savings = calculate_xai_operational_savings(100000, escalated_n)
saved_hrs = savings["compute_time_saved_hours"]
red_pct = savings["compute_reduction_pct"]
print(f"Compute time saved: {saved_hrs:.2f} hours ({red_pct:.1f}% reduction)")


In [ ]:
# --- Principled Gating & Streaming Benchmarks (Shannon Entropy & Split-Conformal) ---
print("[*] Testing Normalized Shannon Entropy Gating vs. Split-Conformal Sets...")
entropy_ctrl = CascadedNIDSController(tier1_model, gating_mode="entropy", entropy_threshold=0.80)
entropy_res = entropy_ctrl.evaluate_traffic_stream(X_val, y_true=y_val)
print(f"Entropy Gating -> Resolved: {entropy_res['tier1_resolved_pct']:.2f}%, Escalated: {entropy_res['tier2_escalated_pct']:.2f}%")

conformal_ctrl = CascadedNIDSController(tier1_model, gating_mode="conformal", conformal_alpha=0.05)
conformal_ctrl.calibrate_conformal_quantile(X_val[:min(2000, len(X_val))], y_val[:min(2000, len(y_val))])
conformal_res = conformal_ctrl.evaluate_traffic_stream(X_val, y_true=y_val)
print(f"Conformal Gating -> Resolved: {conformal_res['tier1_resolved_pct']:.2f}%, Escalated: {conformal_res['tier2_escalated_pct']:.2f}%")

# Single-flow streaming latency evaluation (batch=1 with warmup)
print("
[*] Benchmarking Single-Flow (batch=1) Streaming Latency with 1,000-flow warmup...")
streaming_bench = entropy_ctrl.benchmark_streaming_latency(X_val, n_warmup=1000, n_eval=min(5000, len(X_val)))
print(f"  Mean: {streaming_bench.get('mean_us', 0):.2f} us | P50: {streaming_bench.get('p50_us', 0):.2f} us | P90: {streaming_bench.get('p90_us', 0):.2f} us | P99: {streaming_bench.get('p99_us', 0):.2f} us")
